In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append("..")
import numpy as np
from sklearn.model_selection import train_test_split
from config import *
import pandas as pd
from utils.balanced_builders_hdf import *
from Composition.pipelines import *
from Composition.run_experiment import *

In [2]:
balanced_ids = np.load(balanced_ids , allow_pickle=True)
ids = np.load(ids_gaussian_new , allow_pickle=True)
compositions = np.load(comp_path, allow_pickle=True)
crystal_systems = np.load(cs_path, allow_pickle=True)

id_to_index = {id_: i for i, id_ in enumerate(ids)}
balanced_indices = np.array(
    [id_to_index[id_] for id_ in balanced_ids],
    dtype=np.int64
)
balanced_ids_names = ids[balanced_indices]
compositions = compositions[balanced_indices]
balanced_cs = crystal_systems[balanced_indices]

In [3]:
compositions, X_train, X_test = load_composition_dataset(compositions)

In [4]:
out_mag= run_magpie_pipeline(X_train, X_test,top_k=5,save_path=MAGPIE_EMBEDDINGS)

In [5]:
out_elem = run_qwen_element_pipeline( X_train, X_test, save_path=ELEMENT_EMBEDDINGS)

[ElementEncoder] Loaded cache from qwen_element_cache.pkl


In [ ]:
out_comp= run_qwen_composition_pipeline(X_train, X_test, save_path=COMP_EMBEDDINGS)

In [3]:
# ------------------------------------------------------------
# 1) Qwen vs Qwen (representation analysis)
# ------------------------------------------------------------
qwen_vs_qwen = os.path.join(result_compare, "qwen_vs_qwen")
qwen_results = run_experiment(
    elem_path=ELEMENT_EMBEDDINGS,
    comp_path=COMP_EMBEDDINGS,
    out_dir=qwen_vs_qwen,
    top_k=5
)

print("Qwen vs Qwen results:")
print(qwen_results)

save_results_table(
    {
        "top_k": qwen_results["top_k"],
        "Element acc (mean)": qwen_results["element_accuracy_mean"],
        "Element acc (std)": qwen_results["element_accuracy_std"],
        "Composition acc (mean)": qwen_results["composition_accuracy_mean"],
        "Composition acc (std)": qwen_results["composition_accuracy_std"],
        "Overlap@k": qwen_results["mean_overlap"],
        "Space corr": qwen_results["space_correlation"],
        "Clustering AMI": qwen_results["clustering_ami"],
    },
    out_dir=qwen_vs_qwen ,
    filename="qwen_vs_qwen")

Qwen vs Qwen results:
{'top_k': 5, 'space_correlation': 0.24030651152133942, 'element_accuracy_mean': 0.9853333333333332, 'composition_accuracy_mean': 0.9006666666666666, 'element_accuracy_std': 0.07154175160158034, 'composition_accuracy_std': 0.1962979594618571, 'mean_overlap': 0.18533333333333335, 'clustering_ami': 0.15112867186746815}


In [4]:
qwen_vs_magpie = os.path.join(result_compare, "qwen_vs_magpie")
magpie_results = compare_magpie_vs_qwen(
    magpie_path=MAGPIE_EMBEDDINGS,
    qwen_elem_path=ELEMENT_EMBEDDINGS,
    #out_dir="results/qwen_vs_magpie",
    top_k=5,
    n_queries=1000
)

print("\nMagpie vs Qwen results:")
print(magpie_results)
save_results_table(
    {
        "top_k": 5,
        "Magpie acc (mean)": magpie_results["magpie_accuracy_mean"],
        "Magpie acc (std)": magpie_results["magpie_accuracy_std"],
        "Qwen acc (mean)": magpie_results["qwen_accuracy_mean"],
        "Qwen acc (std)": magpie_results["qwen_accuracy_std"],
    },
    out_dir=qwen_vs_magpie,
    filename="qwen_vs_magpie"
)


Magpie vs Qwen results:
{'magpie_accuracy_mean': 0.8498, 'magpie_accuracy_std': 0.23520195577418143, 'qwen_accuracy_mean': 0.9852000000000001, 'qwen_accuracy_std': 0.08775511381110505}


In [ ]:

out_path = os.path.join(
    result_compare,
    "qwen_vs_magpie",
    "tsne_magpie_vs_qwen.pdf"
)

plot_tsne_magpie_vs_qwen_pca(
    MAGPIE_EMBEDDINGS,
    ELEMENT_EMBEDDINGS,
    out_path=out_path,
    n_samples=5000,
    pca_dim=50,
    perplexity=30,
    random_state=42,
)
